# TP Redis : cas d'utilisation

# Préparer l'environnement

Depuis la racine du dépôt, placez-vous dans `docker`. Démarrez le service et vérifiez son état :

:::bash
cd docker
docker compose up -d redis
docker compose ps redis
:::

## Exercice 1 - Cas d’utilisation STRING : gestion des sessions utilisateur

### Mise en situation

Cet exercice met en pratique le type STRING de Redis dans le contexte d’une application Web.

Dans les applications Web modernes, lorsqu’un utilisateur se connecte avec son identifiant et son mot de passe, l’application vérifie son identité. Si l’authentification réussit, l’application peut lui fournir un token de session.

Un token peut être vu comme un laissez-passer temporaire permettant à l’application de reconnaître un utilisateur déjà authentifié. Il joue en quelque sorte le rôle d’un badge d’accès : une fois authentifié, l’utilisateur n’a pas besoin de transmettre son mot de passe à chaque nouvelle requête.

Lors des requêtes suivantes, le navigateur transmet ce token à l’application. Celle-ci peut alors vérifier à quel utilisateur il correspond et si sa session est toujours valide.

Pour permettre un accès rapide à ces informations, Redis peut être utilisé pour stocker temporairement les tokens de session. Son mécanisme d’expiration (TTL) permet également de supprimer automatiquement un token lorsque sa durée de validité est dépassée.

### Exemple

Vous développez une application web. Après authentification, l'utilisateur reçoit un token sous la forme : `a3f5e8d9c2f1-demo`.
La session est alors enregistrée sous la clé `tp:session:12345` ou 12345 est l'identifiant de l'utilisateur avec une expiration automatique après 30 minutes, soit 1 800 secondes.

:::{important} Important
Le token utilisé ici est fictif et sert uniquement à l'exercice. Dans une application réelle, ne stockez jamais un mot de passe en clair et ne diffusez pas les tokens dans les journaux.
:::

### Connexion à Redis

Etablissez dans un premier la connexion à notre service redis depuis le notebook :

In [1]:
import redis

# Le nom d'hôte 'redis' est utilisé depuis le conteneur JupyterLab.
r = redis.Redis(
    host='172.22.23.76',
    port=7010,
    db=0,
    decode_responses=True
)

print('Connexion Redis :', r.ping())

Connexion Redis : True


### A vous de jouer !

#### Créer une cle session

Après authentification, stockez le token dans Redis pendant 30 minutes. Utilisez `SET` avec l'option `EX`.

Syntaxe utile :

```text
SET clé valeur EX durée_en_secondes
```

Le lien pour l'API python : https://redis.io/docs/latest/commands/set/

In [2]:
# Écrire ici le code Python permettant de créer
# tp:session:12345 avec le token a3f5e8d9c2f1-demo
# et une expiration de 1800 secondes.

# Votre code :
session_key = 'tp:session:12345'
session_token = 'a3f5e8d9c2f1-demo'
result = r.set(name= session_key, value=session_token, ex=1800)
print('Session créée :', result)

Session créée : True


`SET` retourne `True` en Python lorsque la valeur a été enregistrée. L'expiration est attachée à la clé et sera gérée automatiquement par Redis.

:::{admonition} Solution !
:class: dropdown

```python
session_key = 'tp:session:12345'
session_token = 'a3f5e8d9c2f1-demo'

# EX 1800 associe immédiatement une durée de vie à la clé.
result = r.set(session_key, session_token, ex=1800)
print('Session créée :', result)
```

`SET` enregistre la valeur et l'option `ex=1800` définit une expiration de 30 minutes.
:::

#### Récupérer le token

Lorsqu'une requête arrive avec l'identifiant de session, récupérez le token associé avec `GET`.

Le lien pour l'API python : https://redis.io/docs/latest/commands/get/

In [44]:

# Récupérer la valeur de session_key dans Redis.

# Votre code :


Session créée : True
Token récupéré : a3f5e8d9c2f1-demo


:::{admonition} Solution !
:class: dropdown

```python
retrieved_token = r.get(session_key)
print('Token récupéré :', retrieved_token)
```

:::

#### Contrôler le TTL

Vérifiez le nombre de secondes restantes avant l'expiration avec `TTL`. La valeur exacte diminue à chaque instant, mais elle doit être positive et inférieure ou égale à 1 800 : https://redis.io/docs/latest/commands/ttl/

In [ ]:

# Afficher le TTL de la session et vérifier qu'il est positif.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
ttl_seconds = r.ttl(session_key)
print('TTL restant :', ttl_seconds, 'secondes')
```

`TTL` retourne le nombre approximatif de secondes restantes. Une valeur positive confirme que la session possède une expiration active.
:::

`TTL` retourne `-1` lorsqu'une clé existe sans expiration et `-2` lorsque la clé n'existe pas.

#### Renouveler la session

L'utilisateur est toujours actif. Renouvelez donc la durée de vie de la session à 30 minutes avec `EXPIRE` : https://redis.io/docs/latest/commands/expire/


In [ ]:
# Renouveler le TTL de session_key à 1800 secondes,
# puis afficher le nouveau TTL.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
renewed = r.expire(session_key, 1800, xx=True)
print('Expiration renouvelée :', renewed)
print('Nouveau TTL :', r.ttl(session_key), 'secondes')
```

`EXPIRE` modifie la durée de vie d'une clé existante. La commande retourne `True` si la clé existe.
L'option NX ré-initialise l'expiration si la clé a déjà une durée d'expiration.
:::

`EXPIRE` retourne `True` si la clé existe et si son expiration a été modifiée. Il retourne `False` si la clé n'existe pas.

#### Déconnexion

Lors de la déconnexion, supprimez immédiatement la session avec `DEL`, puis vérifiez qu'elle n'est plus accessible : https://redis.io/docs/latest/commands/del/

In [ ]:

# Supprimer la session, afficher le nombre de clés supprimées,
# puis vérifier que GET retourne None.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
deleted_count = r.delete(session_key)
print('Nombre de sessions supprimées :', deleted_count)
print('Session après déconnexion :', r.get(session_key))
assert deleted_count == 1
assert r.get(session_key) is None
```

`DEL` supprime immédiatement la session. Après suppression, `GET` retourne `None` car la clé n'existe plus.
:::

#### Question d'approfondissement

Comment empêcher l'écrasement d'une session existante lors de sa création ? Consultez `help set` ou la documentation de Redis et recherchez l'option `NX` :https://redis.io/docs/latest/commands/set/

In [ ]:

# Proposer le code Python utilisant SET avec NX.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
new_session_key = 'tp:session:67890'

created = r.set(
    new_session_key,
    'token-demo-67890',
    ex=1800,
    nx=True
)

if created:
    print('Session créée')
else:
    print('Session déjà en cours')
```

Avec `nx=True`, `SET` n'écrase pas une clé existante. La méthode retourne `True` si la clé est créée et `None` si elle existe déjà.
:::

## Exercice 2 - Cas d'utilisation LIST : une file de tâches

### Mise en situation

Votre application doit traiter des tâches de manière asynchrone : traitement de fichiers, envoi d'e-mails ou génération de rapports. Les tâches sont déposées dans une file Redis et des workers les traitent dans leur ordre d'arrivée (cf airflow).

La clé `tp:tasks:queue` représente la file d'attente. Chaque valeur est une tâche identifiée par une chaîne de caractères.

Le fonctionnement attendu est **FIFO** (*First In, First Out*) : la première tâche ajoutée doit être la première tâche traitée. Les doublons sont autorisés, car deux tâches identiques peuvent être demandées à des moments différents.

### A vous de jouer !

#### Ajouter quatre tâches à la fin de la file avec `RPUSH` :

https://redis.io/docs/latest/commands/rpush/


In [ ]:
# Ajouter les tâches suivantes dans la file :
# traiter_fichier_01, envoyer_email_02,
# generer_rapport_03 et nettoyer_donnees_04.
# Utiliser RPUSH et la clé tp:tasks:queue.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
queue_key = 'tp:tasks:queue'
r.delete(queue_key)

r.rpush(
    queue_key,
    'traiter_fichier_01',
    'envoyer_email_02',
    'generer_rapport_03',
    'nettoyer_donnees_04'
)
print('Tâches ajoutées :', r.lrange(queue_key, 0, -1))
```

`RPUSH` ajoute les tâches à droite de la liste et conserve leur ordre d'arrivée. La suppression initiale garantit que l'exercice peut être rejoué proprement.
:::

#### Consulter toutes les tâches en attente avec `LRANGE` ;
https://redis.io/docs/latest/commands/lrange/

In [ ]:
# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
# LRANGE 0 -1 retourne toute la liste, du premier au dernier élément.
pending_tasks = r.lrange(queue_key, 0, -1)
print('Tâches en attente :', pending_tasks)
```

`LRANGE` lit une portion de la liste sans la modifier. Les indices `0` et `-1` désignent respectivement le premier et le dernier élément.
:::

#### Récupérer et supprimer la première tâche avec `LPOP` ;
https://redis.io/docs/latest/commands/lpop/

In [ ]:
# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
# LPOP retire et retourne l'élément situé à gauche de la liste.
processed_task = r.lpop(queue_key)
print('Tâche récupérée par le worker :', processed_task)
assert processed_task == 'traiter_fichier_01'
```

`LPOP` réalise l'opération de traitement FIFO : il retourne puis retire la tâche la plus ancienne.
:::

#### consulter la prochaine tâche sans la supprimer avec `LINDEX` ;
https://redis.io/docs/latest/commands/lindex/

In [ ]:
# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
# LINDEX lit un élément à un indice sans le retirer de la liste.
next_task = r.lindex(queue_key, 0)
print('Prochaine tâche :', next_task)
assert next_task == 'envoyer_email_02'
```

`LINDEX 0` consulte la prochaine tâche sans la supprimer. Elle reste donc disponible pour le worker qui la traitera ensuite.
:::

#### Afficher le contenu final de la file :
https://redis.io/docs/latest/commands/lrange/

In [ ]:
# envoyer_email_02, generer_rapport_03 et nettoyer_donnees_04.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
remaining_tasks = r.lrange(queue_key, 0, -1)
print('File finale :', remaining_tasks)
assert remaining_tasks == [
    'envoyer_email_02',
    'generer_rapport_03',
    'nettoyer_donnees_04'
]

# Nettoyage de la clé utilisée par l'exercice.
r.delete(queue_key)
```

Après `LPOP`, la première tâche a disparu et les trois tâches suivantes restent dans le même ordre FIFO. Le nettoyage permet de rejouer l'exercice sans conserver de données résiduelles.
:::

Dans une application réelle, plusieurs workers peuvent ainsi récupérer les tâches d'une file. Un très bon exemple est apache airflow.

## Exercice 3 - Cas d'utilisation SET : gérer les utilisateurs connectés

### Mise en situation

Votre application doit suivre les utilisateurs actuellement connectés au site. Chaque utilisateur est identifié par un identifiant unique.

La clé `tp:online:users` représentera l'ensemble des utilisateurs connectés. Un utilisateur ne doit apparaître qu'une seule fois, même si plusieurs événements de connexion sont reçus.

#### Ajouter les utilisateurs `42`, `57`, `81`, `104` et `205` puis ajouter à nouveau l'utilisateur `42` et observer le résultat :
https://redis.io/docs/latest/commands/sadd/



In [ ]:
# Ajouter les utilisateurs 42, 57, 81, 104 et 205 dans
# l'ensemble tp:online:users avec SADD.
# Ajouter ensuite 42 une seconde fois et observer le résultat.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
online_users_key = 'tp:online:users'
r.delete(online_users_key)

added_count = r.sadd(online_users_key, 42, 57, 81, 104, 205)
duplicate_count = r.sadd(online_users_key, 42)
print('Utilisateurs ajoutés :', added_count)
print('Ajout du doublon 42 :', duplicate_count)
```

`SADD` retourne le nombre de nouveaux éléments ajoutés. Le second ajout de `42` retourne `0`, car un `SET` ne contient pas de doublons.
:::

#### vérifier si `42` et `999` sont connectés :
https://redis.io/docs/latest/commands/sismember/


In [ ]:
# Vérifier si les utilisateurs 42 et 999 sont connectés avec SISMEMBER.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
is_user_42_online = r.sismember(online_users_key, 42)
is_user_999_online = r.sismember(online_users_key, 999)
print('Utilisateur 42 connecté :', is_user_42_online)
print('Utilisateur 999 connecté :', is_user_999_online)

```

`SISMEMBER` retourne `True` si la valeur appartient au `SET` et `False` dans le cas contraire. Cette vérification est rapide, même si l'ensemble contient beaucoup d'utilisateurs.
:::

#### afficher tous les utilisateurs connectés ;
https://redis.io/docs/latest/commands/smembers/
https://redis.io/docs/latest/commands/scard/

In [ ]:
# Afficher tous les utilisateurs connectés avec SMEMBERS,
# puis compter leur nombre avec SCARD.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
connected_users = r.smembers(online_users_key)
connected_count = r.scard(online_users_key)
print('Utilisateurs connectés :', connected_users)
print('Nombre d utilisateurs connectés :', connected_count)

```

`SMEMBERS` retourne les membres du `SET`. L'ordre n'est pas garanti. `SCARD` retourne le nombre d'éléments uniques présents dans l'ensemble.
:::

#### simuler la déconnexion de l'utilisateur `42` et vérifier le résultat.
https://redis.io/docs/latest/commands/srem/

In [ ]:
# Supprimer l'utilisateur 42 avec SREM, vérifier qu'il n'est plus
# connecté, puis afficher le nombre final d'utilisateurs.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
removed_count = r.srem(online_users_key, 42)
print('Utilisateurs supprimés :', removed_count)
print('Utilisateur 42 connecté :', r.sismember(online_users_key, 42))
print('Nombre final d'utilisateurs :', r.scard(online_users_key))
assert removed_count == 1
assert r.sismember(online_users_key, 42) is False
assert r.scard(online_users_key) == 4

```

`SREM` supprime un membre du `SET` et retourne `1` si la suppression a eu lieu. Après la déconnexion de `42`, il reste quatre utilisateurs connectés.
:::

Ce modèle convient pour gérer des utilisateurs connectés, des tags, des groupes ou des droits. Il permet également d'explorer des opérations ensemblistes comme `SINTER`, `SUNION` et `SDIFF`.

## Exercice 4 - Cas d'utilisation ZSET : classement de joueurs

### Mise en situation

Votre application de jeu en ligne doit afficher le classement des joueurs selon leur score. La clé `tp:game:leaderboard` représente le classement.

Dans un `ZSET`, chaque joueur est un membre associé à un score numérique. Redis maintient automatiquement l'ordre des joueurs selon leur score et un joueur ne peut apparaître qu'une seule fois.

#### ajouter les joueurs `Alice` avec 1 250 points, `Bob` avec 980 points, `Charlie` avec 1 100 points et `David` avec 870 points puis afficher les tous :

https://redis.io/docs/latest/commands/zadd/

https://redis.io/docs/latest/commands/zrevrange/


In [ ]:
# Ajouter les quatre joueurs avec leurs scores dans le ZSET
# tp:game:leaderboard avec ZADD.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
leaderboard_key = 'tp:game:leaderboard'
r.delete(leaderboard_key)

r.zadd(leaderboard_key, {
    'Alice': 1250,
    'Bob': 980,
    'Charlie': 1100,
    'David': 870
})
print('Classement créé :', r.zrevrange(leaderboard_key, 0, -1, withscores=True))
```

`ZADD` associe un score à chaque membre. `ZREVRANGE` permet ici de les lire du meilleur au moins bon.
:::

#### afficher le top 3 avec les scores, du meilleur au moins bon ;


In [ ]:

# Afficher le top 3 avec les scores, du meilleur au moins bon,
# en utilisant ZREVRANGE.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
top_three = r.zrevrange(leaderboard_key, 0, 2, withscores=True)
print('Top 3 :', top_three)
assert [player for player, score in top_three] == ['Alice', 'Charlie', 'Bob']
```

Les indices `0` à `2` sélectionnent trois membres. `withscores=True` ajoute le score de chaque joueur au résultat.
:::

#### obtenir la position de `Bob` dans le classement :

https://redis.io/docs/latest/commands/zrevrank/


In [ ]:
# Obtenir la position de Bob avec ZREVRANK.
# La position attendue est exprimée à partir de 0.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
bob_rank = r.zrevrank(leaderboard_key, 'Bob')
print('Position de Bob :', bob_rank)
assert bob_rank == 2
```

`ZREVRANK` donne le rang dans l'ordre décroissant des scores. Le premier joueur a le rang `0`, donc Bob est ici en troisième position.
:::

#### mettre à jour le score de `Bob` à 1 500 points, vérifier sa nouvelle position et afficher le classement final.

In [ ]:
 - étape 4
# Mettre à jour le score de Bob à 1500 points
# vérifier sa nouvelle position et afficher le classement final.

# Votre code :

:::{admonition} Solution !
:class: dropdown

```python
updated_count = r.zadd(leaderboard_key, {'Bob': 1500})
new_bob_rank = r.zrevrank(leaderboard_key, 'Bob')
final_leaderboard = r.zrevrange(leaderboard_key, 0, -1, withscores=True)

print('Membre mis à jour :', updated_count)
print('Nouvelle position de Bob :', new_bob_rank)
print('Classement final :', final_leaderboard)

```

`ZADD` met à jour le score d'un membre existant. Bob passe en première position avec le rang `0`. Le classement est réordonné automatiquement.
:::

Ce modèle est adapté aux classements, aux tops, aux files prioritaires et aux métriques ordonnées.

## Démonstration - Cas d'utilisation Pub/Sub : un tchat avec des clients séparés

Dans cet exercice, Redis est le serveur central. Deux conteneurs distincts jouent le rôle de clients en ligne de commande, tandis que le notebook Python jouera successivement le rôle de publieur et d'abonné. Les trois clients se connectent au serveur Redis via le nom DNS `redis` sur le réseau Docker `datalab`.

Les messages ne sont transmis qu'aux abonnés connectés au moment de la publication. Redis Pub/Sub ne conserve pas d'historique.

### Préparer les conteneurs

Depuis le répertoire `docker` du dépôt, démarrer le serveur Redis, les deux conteneurs clients et JupyterLab :

```bash
docker compose up -d redis redis-publisher redis-subscriber jupyterlab
```

### Démonstration A : le notebook publie, le client Docker reçoit

Dans un terminal, ouvrir le client abonné :

```bash
docker compose exec -it redis-subscriber redis-cli -h redis -p 6379
```

Dans le prompt `redis-cli`, abonnez vous au canal **tp:chat:general** :

```text
SUBSCRIBE tp:chat:general
```

La souscription déclenche un **listener** pour capturer tous les messages qui sont transmis au canal **tp:chat:general** et explique les messages continus :

```
Reading messages... (press Ctrl-C to quit or any key to type command)
```

Laisser ce terminal ouvert. 

Nous allons à présent utiliser la librairie python redis pour publier un message dans notre canal **tp:chat:general**.
:::{note} Note
Pour plus de détail sur l'API Python redis : https://redis.io/docs/latest/develop/clients/redis-py/
:::

In [13]:
import redis

r = redis.Redis(
    host='172.22.23.76',
    port=7010,
    db=0,
    decode_responses=True
)
print('Connexion Redis :', r.ping())

chat_channel = 'tp:chat:general'
message_text = 'Jupyter : Bonjour a tous !'
subscriber_count = r.publish(chat_channel, message_text)
print('Nombre d abonnés ayant reçu le message :', subscriber_count)

Connexion Redis : True
Nombre d abonnés ayant reçu le message : 1


#### Observer dans le client `redis-subscriber`

:::{important} A noter
On peut publier dans un canal même si on y a pas souscrit :-(
    

Réalisons à présent expérience inverse.

### Démonstration B : le client Docker publie, le notebook reçoit

Utilisons maintenant notre notebook pour s'abonner à notre canal **tp:chat:general**.
On va pour cela créer à partir de notre connexion à redis (r) un objet spécial (PubSub) conçu pour gérer les abonnements et la publication des messages.

In [34]:

chat_subscriber = r.pubsub()
type(chat_subscriber)

redis.client.PubSub

Pour ensuite déclarer notre abonnement à notre canal **tp:chat:general**

In [39]:
chat_channel = 'tp:chat:general'
chat_subscriber.subscribe(chat_channel)


On peut lister les canaux dans lesquels on écoute :

In [41]:
chat_subscriber.channels

{'tp:chat:general': None, 'tp:chat:test': None}

Pour récuperer les messages publiés dans notre canal il suffit d'utiliser la méthode get_message()

In [40]:
subscription_message = chat_subscriber.get_message(timeout=1)
print('Confirmation d abonnement :', subscription_message)

Confirmation d abonnement : {'type': 'subscribe', 'pattern': None, 'channel': 'tp:chat:test', 'data': 2}


Notre Notebook est abonné à notre canal et on va à présent ouvrir une client redis-publisher dans un second terminal avec la commande suivante :

```bash
docker compose exec -it redis-publisher redis-cli -h redis -p 6379
```

On est prêt à publier un message depuis  redis-publisher mais pour pouvoir capturer les messages en continu nous allons devoir demander à notre notebook de passer en mode ecoute avec le code suivant :

In [30]:
# cellule en écoute

for message in chat_subscriber.listen():
    if message["type"] == "message":
        print(
            f"Canal   : {message['channel']}\n"
            f"Message : {message['data']}\n"
        )

Canal   : tp:chat:general
Message : Redis_publisher : Il y a quelqu un !



KeyboardInterrupt: 

On peut à partir du second terminal publier un message en copiant la commande suivante :

```
PUBLISH "Redis_publisher : Bonjour a tous !"
```

Le message devrait être reçu à la fois par le notebook et par l’abonné Docker (terminal1 / redis-subscriber).

Si on ne veut plus recevoir les messages il suffit de unsubscribe.
Dans notre terminal1 (redis-subscriber), exécuter la commande 

```
UNSUBSCRIBE tp:chat:general
```

Dans le terminal2 (Redis_publisher) publier le message suivant :
```
PUBLISH tp:chat:general "Redis_publisher : Il y a quelqu un !"
```

Vous devriez voir le message dans le notebook mais plus sur  terminal1 (redis-subscriber)

Pour désabonner notre notebook, il faudra dans un premier temps arrêter la cellule en écoute pour ensuite exécuter le code suivant :

In [31]:
chat_subscriber.unsubscribe(chat_channel)

Désabonnement : {'type': 'unsubscribe', 'pattern': None, 'channel': 'tp:chat:general', 'data': 0}


On peut vérifier que notre notebook n'écoute plus sur notre canal **tp:chat:general** en testant la collecte de messages.

In [32]:
unsubscribe_message = chat_subscriber.get_message(timeout=1)
print('Abonnement :', unsubscribe_message)

Désabonnement : None


Pour ne plus gérer les abonnements et la publication des messages, on peut clore notre objet PubSub

In [33]:
chat_subscriber.close()

## Bilan de l'exercice Pub/Sub

Vous avez fait communiquer trois clients indépendants par l'intermédiaire d'un serveur Redis : le notebook Python, `redis-publisher` et `redis-subscriber`. Les conteneurs clients ne stockent pas les messages ; ils se connectent au service `redis` par son nom sur le réseau Docker partagé `datalab`.

- `SUBSCRIBE` abonne un client à un canal ;
- `PUBLISH` diffuse un message aux abonnés actifs ;
- un même message peut être reçu par plusieurs clients ;
- `UNSUBSCRIBE` termine un abonnement ;
- Pub/Sub ne persiste pas les messages et ne fournit pas d'historique.

Pour conserver les messages d'un tchat, il faudrait associer Pub/Sub à une `LIST`, à un `STREAM` Redis ou à une autre solution de stockage.